# ✈️ PIPELINE ETL: DADOS DA ANAC PARA O POSTGRESQL (REFATORADO)

Versão limpa e refatorada do pipeline de dados, organizada em 4 passos lógicos:
- **Passo 1:** Importação das bibliotecas (`pandas` e `psycopg2`)
- **Passo 2:** Extração dos dados (leitura do arquivo JSON)
- **Passo 3:** Transformação dos dados (filtragem e remoção de acentos)
- **Passo 4:** Carga no PostgreSQL (limpeza com TRUNCATE, inserção em lote e validação)


In [5]:
# ==============================================================================
# PASSO 1: IMPORTAÇÃO DAS BIBLIOTECAS
# ==============================================================================

import pandas as pd     # Manipulação e estruturação de tabelas (DataFrames)
import psycopg2         # Driver conector com o banco de dados PostgreSQL


In [6]:
# ==============================================================================
# PASSO 2: EXTRAÇÃO DOS DADOS (LEITURA DO JSON)
# ==============================================================================

# Caminho do arquivo JSON (o prefixo 'r' define raw string e evita erros de barras)
caminho_do_arquivo = r"/Users/analeia/Desktop/Engenharia de Dados/V_OCORRENCIA_AMPLA.json"

# Lê o JSON e carrega na memória como tabela. utf-8-sig preserva acentos corretamente
db = pd.read_json(caminho_do_arquivo, encoding="utf-8-sig")


In [7]:
# ==============================================================================
# PASSO 3: TRANSFORMAÇÃO (FILTRAR E PADRONIZAR NOMES DE COLUNAS)
# ==============================================================================

# 1. Lista apenas as 7 colunas necessárias para o banco de dados
colunas = [
    "Numero_da_Ocorrencia",         # Identificador único da ocorrência
    "Classificacao_da_Ocorrência",  # Gravidade (ainda com acento no arquivo original)
    "Data_da_Ocorrencia",           # Data do evento
    "Municipio",                    # Município/cidade
    "UF",                           # Estado
    "Regiao",                       # Região do Brasil
    "Nome_do_Fabricante"            # Fabricante da aeronave
]

# Proteção contra reexecução: se a célula for executada novamente, evita KeyError
if 'Classificacao_da_Ocorrência' not in db.columns and 'Classificacao_da_Ocorrencia' in db.columns:
    colunas[1] = 'Classificacao_da_Ocorrencia'

# Filtra o DataFrame mantendo apenas as colunas da lista
db = db[colunas]

# 2. Remove o acento da coluna (boa prática em banco de dados relacional)
if 'Classificacao_da_Ocorrência' in db.columns:
    db = db.rename(columns={'Classificacao_da_Ocorrência': 'Classificacao_da_Ocorrencia'})


In [8]:
# ==============================================================================
# PASSO 4: CARGA NO POSTGRESQL E VALIDAÇÃO
# ==============================================================================

# 1. Parâmetros de acesso ao banco PostgreSQL
dbname = 'python'
user = 'analeia'
password = '12345'
host = 'localhost'
port = '5432'

# 2. Abre a conexão com o banco e cria o cursor (executor de comandos SQL)
conn = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cur = conn.cursor()

# 3. Cria a tabela Anac no banco caso ela ainda não exista
cur.execute("""
CREATE TABLE IF NOT EXISTS Anac (
    Numero_da_Ocorrencia INT,
    Classificacao_da_Ocorrencia VARCHAR(50),
    Data_da_Ocorrencia DATE,
    Municipio VARCHAR(50),
    UF VARCHAR(30),
    Regiao VARCHAR(30),
    Nome_do_Fabricante VARCHAR(100)
);
""")

# 4. Limpa dados antigos da tabela para não duplicar registros a cada execução
cur.execute("TRUNCATE TABLE Anac;")

# 5. Percorre cada linha do DataFrame e insere na tabela do banco
for indice, coluna in db.iterrows():
    cur.execute("""
    INSERT INTO Anac (
        Numero_da_Ocorrencia,
        Classificacao_da_Ocorrencia,
        Data_da_Ocorrencia,
        Municipio,
        UF,
        Regiao,
        Nome_do_Fabricante
    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (
        coluna['Numero_da_Ocorrencia'],
        coluna['Classificacao_da_Ocorrencia'],
        coluna['Data_da_Ocorrencia'],
        coluna['Municipio'],
        coluna['UF'],
        coluna['Regiao'],
        coluna['Nome_do_Fabricante']
    ))

# 6. Grava permanentemente todas as alterações no banco de dados
conn.commit()

# 7. Validação: confere se a contagem de linhas no banco bate com o DataFrame
cur.execute("SELECT COUNT(*) FROM Anac;")
total_banco = cur.fetchone()[0]


# 8. Fecha o cursor e a conexão para liberar os recursos do servidor
cur.close()
conn.close()
